# SkyGuard AI — Iteration 12: Genuine IMD AWS Data Foundation

Yeh notebook **official, authenticated IMD AWS observations** collect karta hai. Isme generated weather,
METAR substitution, DWD substitution ya 24-station starter data use nahi hota. Official endpoint se directly
reported temperature, MSLP pressure aur RH preserve hote hain. Har raw response immutable JSON, SHA-256 receipt
aur normalized Parquet ke roop mein Google Drive par save hota hai.

Important: IMD ka documented endpoint current snapshots deta hai, historical training archive nahi. Isliye ek run
sirf ek time-slice hai. Temporal/seasonal model ko honest tarike se train karne ke liye snapshots ko time ke saath
collect karna hoga. Notebook 30-day pilot, 90-day robust temporal aur 365-day seasonal readiness separately batata hai.

## Credentials — API key ko notebook mein paste mat karein

1. IMD API portal mein AWS Data aur AWS Data Mapping access approve karayein.
2. Colab ke left sidebar mein **Secrets (key icon)** kholein.
3. `IMD_API_KEY` aur `IMD_JWT_TOKEN` naam ke do secrets banayein aur notebook access enable karein.
4. JWT expire ho sakta hai; `401 Invalid or expired JWT token` aaye to official portal se renew karein.

Gateway contract independently checked on 12 September 2026: `x-api-key` header plus
`Authorization: Bearer <JWT>`. Secrets kisi output, receipt ya ZIP mein save nahi hote.

Official references: [IMD AWS API](https://api.imd.gov.in/public/api_reference.html),
[IMD API portal](https://api.imd.gov.in/public/index.php).

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'pandas>=2.2,<3', 'numpy>=1.26,<3', 'pyarrow>=16,<24', 'requests>=2.31,<3'])
print('Dependencies ready')

In [ ]:
from pathlib import Path
import json, os, sys, time, shutil
import pandas as pd
from IPython.display import display
from google.colab import drive, userdata

drive.mount('/content/drive')
I12_ROOT = Path('/content/drive/MyDrive/SkyGuard_AI_GPU/experiments/iteration12_genuine_imd_aws_v1')
I12_ROOT.mkdir(parents=True, exist_ok=True)

# 0 = one snapshot. Set e.g. 6 to collect every 15 minutes for six hours in this Colab session.
COLLECT_HOURS = 0
INTERVAL_MINUTES = 15
STATE_ID = None       # None = national endpoint; e.g. 7 = Delhi only
SOURCE_TIMEZONE = 'UTC'  # IMD sample timing is handled under this explicit, auditable contract.
TIMESTAMP_TIMEZONE_CONFIRMED = False  # Set True only after IMD/portal confirms AWS DATE+TIME convention.

IMD_API_KEY = userdata.get('IMD_API_KEY')
IMD_JWT_TOKEN = userdata.get('IMD_JWT_TOKEN')
assert IMD_API_KEY and IMD_JWT_TOKEN, 'Add IMD_API_KEY and IMD_JWT_TOKEN in Colab Secrets first.'
print('Experiment:', I12_ROOT)
print('Secrets loaded in memory; they will not be printed or saved.')

In [ ]:
MODULE_SOURCE = '"""Iteration 12: authenticated, immutable IMD AWS observation collection.\n\nThis module collects the official IMD AWS endpoint only.  It never fabricates\nobservations, silently substitutes another provider, or treats unlabelled rows\nas verified healthy/faulty examples.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport time\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport requests\n\nVERSION = "iteration12-imd-aws-v1"\nAWS_URL = "https://api.imd.gov.in/api/v1/aws_data"\nMAPPING_URL = "https://api.imd.gov.in/api/v1/aws_data_mapping"\nMODEL_INPUTS = ["temperature_c", "pressure_hpa", "relative_humidity_pct"]\nRAW_FIELDS = {\n    "ID": "imd_sensor_id",\n    "CALL_SIGN": "station_id",\n    "DISTRICT": "district",\n    "STATE": "state",\n    "STATION": "station_name",\n    "CURR_TEMP": "temperature_c",\n    "RH": "relative_humidity_pct",\n    "MSLP": "pressure_hpa",\n    "Latitude": "latitude",\n    "Longitude": "longitude",\n}\n\n\ndef utc_now() -> str:\n    return datetime.now(timezone.utc).isoformat()\n\n\ndef sha256_bytes(value: bytes) -> str:\n    return hashlib.sha256(value).hexdigest()\n\n\ndef sha256_file(path: Path) -> str:\n    return sha256_bytes(Path(path).read_bytes())\n\n\ndef _json_rows(payload):\n    """Extract the records list without assuming an undocumented envelope."""\n    if isinstance(payload, list):\n        return payload\n    if isinstance(payload, dict):\n        for key in ("data", "records", "result", "results"):\n            if isinstance(payload.get(key), list):\n                return payload[key]\n        if set(RAW_FIELDS).intersection(payload):\n            return [payload]\n    raise ValueError("IMD response has no supported record list; preserve it and review the API contract")\n\n\ndef fetch_json(url: str, api_key: str, jwt_token: str, *, params=None, timeout=60):\n    if not api_key or not jwt_token:\n        raise ValueError("Both IMD_API_KEY and IMD_JWT_TOKEN are required")\n    headers = {\n        "x-api-key": api_key.strip(),\n        "Authorization": "Bearer " + jwt_token.strip(),\n        "Accept": "application/json",\n        "User-Agent": "SkyGuard-SIH26073-academic-collector/12",\n    }\n    last = None\n    for attempt in range(3):\n        try:\n            response = requests.get(url, headers=headers, params=params, timeout=(20, timeout))\n            if response.status_code == 401:\n                raise PermissionError("IMD rejected the API key/JWT; renew credentials in the official portal")\n            response.raise_for_status()\n            if "json" not in response.headers.get("Content-Type", "").lower():\n                raise ValueError("IMD returned non-JSON content")\n            payload = response.json()\n            return payload, {\n                "url": response.url,\n                "retrieved_at_utc": utc_now(),\n                "http_status": response.status_code,\n                "content_type": response.headers.get("Content-Type"),\n                "last_modified": response.headers.get("Last-Modified"),\n                "payload_sha256": sha256_bytes(response.content),\n                "bytes": len(response.content),\n            }\n        except PermissionError:\n            raise\n        except (requests.RequestException, ValueError) as exc:\n            last = exc\n            if attempt < 2:\n                time.sleep(2**attempt)\n    raise RuntimeError(f"Official IMD request failed after retries: {last}")\n\n\ndef normalize_aws(payload, *, source_timezone="UTC", collected_at_utc=None):\n    rows = _json_rows(payload)\n    if not rows:\n        raise ValueError("IMD AWS response contains zero records")\n    raw = pd.DataFrame(rows)\n    missing = [key for key in ("CALL_SIGN", "DATE", "TIME", "CURR_TEMP", "RH", "MSLP") if key not in raw]\n    if missing:\n        raise ValueError(f"IMD AWS schema is missing required fields: {missing}")\n    out = pd.DataFrame(index=raw.index)\n    for source, target in RAW_FIELDS.items():\n        out[target] = raw[source] if source in raw else pd.NA\n    source_text = raw["DATE"].astype(str).str.strip() + " " + raw["TIME"].astype(str).str.strip()\n    local = pd.to_datetime(source_text, errors="coerce")\n    if source_timezone.upper() == "UTC":\n        timestamp = local.dt.tz_localize("UTC", ambiguous="NaT", nonexistent="NaT")\n    else:\n        timestamp = local.dt.tz_localize(source_timezone, ambiguous="NaT", nonexistent="NaT").dt.tz_convert("UTC")\n    out["timestamp_utc"] = timestamp\n    out["source_date"] = raw["DATE"].astype(str)\n    out["source_time"] = raw["TIME"].astype(str)\n    out["source_timezone_contract"] = source_timezone\n    out["collected_at_utc"] = collected_at_utc or utc_now()\n    for col in MODEL_INPUTS + ["latitude", "longitude"]:\n        out[col] = pd.to_numeric(out[col], errors="coerce")\n    for col in ["station_id", "imd_sensor_id", "station_name", "district", "state"]:\n        out[col] = out[col].fillna("").astype(str).str.strip()\n    out["temperature_reported_directly"] = True\n    out["humidity_reported_directly"] = True\n    out["pressure_source"] = "IMD_MSLP"\n    out["source"] = "IMD_AUTHORIZED_AWS_API"\n    out["source_is_genuine"] = True\n    out["generated_or_simulated"] = False\n    out["timestamp_valid"] = out.timestamp_utc.notna()\n    out["coordinates_valid"] = (out.latitude.between(6, 38) & out.longitude.between(68, 98))\n    out["primary_complete"] = out[MODEL_INPUTS].notna().all(axis=1)\n    out["broad_physical_range_ok"] = (\n        out.temperature_c.between(-60, 65)\n        & out.pressure_hpa.between(850, 1100)\n        & out.relative_humidity_pct.between(0, 100)\n    )\n    out["eligible_for_unlabelled_baseline"] = (\n        out.timestamp_valid & out.coordinates_valid & out.primary_complete & out.broad_physical_range_ok\n    )\n    out["ground_truth_fault"] = -1\n    out["label_status"] = "unlabelled_observation"\n    out = out.sort_values(["station_id", "timestamp_utc"], kind="stable")\n    return out.drop_duplicates(["station_id", "timestamp_utc"], keep="last").reset_index(drop=True)\n\n\ndef archive_snapshot(root, payload, receipt, *, source_timezone="UTC"):\n    root = Path(root)\n    retrieved = pd.Timestamp(receipt["retrieved_at_utc"])\n    stamp = retrieved.strftime("%Y%m%dT%H%M%SZ")\n    raw_dir = root / "raw" / retrieved.strftime("%Y/%m/%d")\n    raw_dir.mkdir(parents=True, exist_ok=True)\n    raw_path = raw_dir / f"imd_aws_{stamp}.json"\n    encoded = json.dumps(payload, ensure_ascii=False, separators=(",", ":")).encode("utf-8")\n    payload_hash = sha256_bytes(encoded)\n    if raw_path.exists() and sha256_file(raw_path) != payload_hash:\n        raise FileExistsError(f"Immutable snapshot collision: {raw_path}")\n    raw_path.write_bytes(encoded)\n    frame = normalize_aws(payload, source_timezone=source_timezone, collected_at_utc=receipt["retrieved_at_utc"])\n    frame["source_snapshot_hash"] = payload_hash\n    clean_dir = root / "normalized" / retrieved.strftime("%Y/%m/%d")\n    clean_dir.mkdir(parents=True, exist_ok=True)\n    clean_path = clean_dir / f"imd_aws_{stamp}.parquet"\n    if not clean_path.exists():\n        frame.to_parquet(clean_path, index=False)\n    record = {**receipt, "version": VERSION, "raw_path": str(raw_path.relative_to(root)),\n              "raw_saved_sha256": sha256_file(raw_path),\n              "normalized_path": str(clean_path.relative_to(root)), "normalized_sha256": sha256_file(clean_path),\n              "records": len(frame), "stations": int(frame.station_id.nunique()),\n              "complete_primary_records": int(frame.primary_complete.sum()),\n              "credentials_recorded": False, "generated_rows": 0}\n    receipt_path = raw_path.with_suffix(".receipt.json")\n    receipt_path.write_text(json.dumps(record, indent=2), encoding="utf-8")\n    return frame, record\n\n\ndef collect_once(root, api_key, jwt_token, *, state_id=None, source_timezone="UTC"):\n    params = {"sid": str(state_id)} if state_id is not None else None\n    payload, receipt = fetch_json(AWS_URL, api_key, jwt_token, params=params)\n    return archive_snapshot(root, payload, receipt, source_timezone=source_timezone)\n\n\ndef collect_mapping(root, api_key, jwt_token):\n    payload, receipt = fetch_json(MAPPING_URL, api_key, jwt_token)\n    root = Path(root)\n    target = root / "metadata" / "imd_aws_mapping.json"\n    target.parent.mkdir(parents=True, exist_ok=True)\n    encoded = json.dumps(payload, ensure_ascii=False, indent=2).encode("utf-8")\n    target.write_bytes(encoded)\n    receipt.update({"version": VERSION, "path": str(target.relative_to(root)),\n                    "saved_sha256": sha256_file(target), "credentials_recorded": False})\n    target.with_suffix(".receipt.json").write_text(json.dumps(receipt, indent=2), encoding="utf-8")\n    return payload, receipt\n\n\ndef assemble(root):\n    paths = sorted(Path(root).glob("normalized/**/*.parquet"))\n    if not paths:\n        raise ValueError("No normalized IMD snapshots found; run authenticated collection first")\n    frames = [pd.read_parquet(path) for path in paths]\n    data = pd.concat(frames, ignore_index=True)\n    data["timestamp_utc"] = pd.to_datetime(data.timestamp_utc, utc=True, errors="coerce")\n    data = data.sort_values(["station_id", "timestamp_utc", "collected_at_utc"], kind="stable")\n    data = data.drop_duplicates(["station_id", "timestamp_utc"], keep="last").reset_index(drop=True)\n    return data\n\n\ndef readiness(data, *, timestamp_timezone_confirmed=False):\n    d = data.copy()\n    d["timestamp_utc"] = pd.to_datetime(d.timestamp_utc, utc=True, errors="coerce")\n    valid = d.loc[d.eligible_for_unlabelled_baseline.astype(bool)]\n    per_station = valid.groupby("station_id").agg(\n        rows=("timestamp_utc", "size"), first_timestamp=("timestamp_utc", "min"),\n        last_timestamp=("timestamp_utc", "max"), distinct_days=("timestamp_utc", lambda x: x.dt.floor("D").nunique()),\n        latitude=("latitude", "median"), longitude=("longitude", "median"),\n    ).reset_index()\n    if len(per_station):\n        per_station["span_days"] = (per_station.last_timestamp - per_station.first_timestamp).dt.total_seconds().div(86400)\n    else:\n        per_station["span_days"] = pd.Series(dtype=float)\n    complete_fraction = float(d.primary_complete.mean()) if len(d) else 0.0\n    report = {\n        "version": VERSION, "created_at_utc": utc_now(), "source": "authorized IMD AWS API",\n        "rows": len(d), "stations": int(d.station_id.nunique()), "valid_unlabelled_rows": len(valid),\n        "complete_primary_fraction": complete_fraction,\n        "first_timestamp_utc": d.timestamp_utc.min().isoformat() if d.timestamp_utc.notna().any() else None,\n        "last_timestamp_utc": d.timestamp_utc.max().isoformat() if d.timestamp_utc.notna().any() else None,\n        "stations_30_days": int(per_station.distinct_days.ge(30).sum()),\n        "stations_90_days": int(per_station.distinct_days.ge(90).sum()),\n        "stations_365_days": int(per_station.distinct_days.ge(365).sum()),\n        "timestamp_timezone_contracts": sorted(d.source_timezone_contract.dropna().astype(str).unique().tolist()),\n        "timestamp_timezone_confirmed": bool(timestamp_timezone_confirmed),\n        "ready_for_pilot_training": bool(timestamp_timezone_confirmed and len(per_station) >= 50 and per_station.distinct_days.ge(30).sum() >= 50 and complete_fraction >= .90),\n        "ready_for_seasonal_claims": bool(timestamp_timezone_confirmed and len(per_station) >= 50 and per_station.distinct_days.ge(365).sum() >= 50),\n        "verified_real_fault_labels": False, "generated_rows": int(d.generated_or_simulated.astype(bool).sum()),\n        "model_inputs": MODEL_INPUTS,\n        "warning": "Unlabelled official observations are not proof of sensor health; inject faults only in evaluation copies.",\n    }\n    return per_station, report\n'
EXPECTED_SHA256 = 'bcc93d3450686d57eb11d83f8a989753297530051752e9dcffbf5519a17235cb'

import hashlib, importlib
assert hashlib.sha256(MODULE_SOURCE.encode()).hexdigest() == EXPECTED_SHA256
module_path = Path('/content/iteration12_imd_aws.py')
module_path.write_text(MODULE_SOURCE, encoding='utf-8')
sys.path.insert(0, '/content') if '/content' not in sys.path else None
import iteration12_imd_aws as i12
importlib.reload(i12)
print('Loaded', i12.VERSION, '| model inputs:', i12.MODEL_INPUTS)

## 1. Mapping and authenticated source verification

Mapping station identity/coverage ke liye hai; observation rows nahi hai. HTTP/schema mismatch fail-closed hota hai.
Notebook unauthorized response ko public data samajhkar continue nahi karega.

In [ ]:
mapping, mapping_receipt = i12.collect_mapping(I12_ROOT, IMD_API_KEY, IMD_JWT_TOKEN)
print('Official mapping saved; SHA-256:', mapping_receipt['saved_sha256'])
print('Credential values saved:', mapping_receipt['credentials_recorded'])

## 2. Collect official AWS snapshots

Raw API response unchanged store hota hai. Normalized table mein model ke inputs exactly T/P/RH hain.
Wind, rain, dew point aur forecast model features nahi bante. Source ke impossible values overwrite nahi hote;
unhe QC flags ke through training baseline se exclude kiya jata hai.

In [ ]:
def collect_and_show():
    frame, receipt = i12.collect_once(I12_ROOT, IMD_API_KEY, IMD_JWT_TOKEN,
                                      state_id=STATE_ID, source_timezone=SOURCE_TIMEZONE)
    print('Saved:', receipt['raw_path'], '| stations:', receipt['stations'], '| rows:', receipt['records'])
    display(frame[['timestamp_utc','station_id','station_name','state','temperature_c',
                   'pressure_hpa','relative_humidity_pct','primary_complete',
                   'broad_physical_range_ok']].head(20))
    return receipt

receipts = [collect_and_show()]
if COLLECT_HOURS > 0:
    runs = max(1, int(COLLECT_HOURS * 60 / INTERVAL_MINUTES))
    for _ in range(1, runs):
        time.sleep(INTERVAL_MINUTES * 60)
        receipts.append(collect_and_show())
print('Snapshots collected this run:', len(receipts))

## 3. Assemble deduplicated training foundation and readiness gates

Official observation ka matlab verified healthy sensor label nahi hota. Isliye `ground_truth_fault=-1` rahega.
Future evaluation mein controlled faults copy par inject kiye ja sakte hain; raw IMD data kabhi modify nahi hoga.

- 30 days / 50 stations: pilot temporal model allowed.
- 90 days: stronger temporal validation target (reported separately below).
- 365 days / 50 stations: seasonal-pattern claim allowed.
- Real fault accuracy ke liye IMD maintenance/fault labels ya independently adjudicated injected benchmark zaroori hai.

In [ ]:
dataset = i12.assemble(I12_ROOT)
station_quality, readiness = i12.readiness(
    dataset, timestamp_timezone_confirmed=TIMESTAMP_TIMEZONE_CONFIRMED)
processed = I12_ROOT / 'processed'
processed.mkdir(exist_ok=True)
dataset_path = processed / 'imd_aws_observations_deduplicated.parquet'
dataset.to_parquet(dataset_path, index=False)
station_quality.to_csv(I12_ROOT / 'iteration12_station_readiness.csv', index=False)
(I12_ROOT / 'iteration12_data_readiness.json').write_text(json.dumps(readiness, indent=2), encoding='utf-8')
display(pd.Series(readiness))
display(station_quality.sort_values(['distinct_days','rows'], ascending=False).head(30))
print('90-day stations:', int(station_quality.distinct_days.ge(90).sum()))
assert not dataset.generated_or_simulated.astype(bool).any()
assert dataset.source.eq('IMD_AUTHORIZED_AWS_API').all()

## 4. Honest training decision

`ready_for_pilot_training=False` ho to model train karna scientific shortcut hoga. Existing retained Phase 10 model
replace nahi hoga. Data accumulate hone ke baad next training stage station-blocked aur time-blocked splits,
causal temporal features, neighbour age/elevation guards, synthetic fault challenge copies, calibration and incident
false-alarm gates use karega. Accuracy class imbalance se hide na ho, isliye point accuracy primary metric nahi hogi.

In [ ]:
decision = {
    'iteration': 12,
    'data_source': 'authorized IMD AWS API',
    'promote_or_train_now': bool(readiness['ready_for_pilot_training']),
    'seasonal_claim_allowed': bool(readiness['ready_for_seasonal_claims']),
    'retained_production_model_changed': False,
    'reason': ('Data readiness gate passed; create a separately evaluated Iteration 12 model challenger.'
               if readiness['ready_for_pilot_training'] else
               'Continue genuine snapshot collection; insufficient history for honest temporal training.'),
    'required_metrics_next': ['fault incident precision','fault episode recall','false incidents per station-day',
                              'weather-event preservation F1','unseen-station F1','detection latency'],
}
(I12_ROOT / 'iteration12_decision.json').write_text(json.dumps(decision, indent=2), encoding='utf-8')
display(pd.Series(decision))

## 5. Download the small audit package

Raw data Drive par hi rahega. ZIP mein credentials ya huge raw observations nahi honge. Team/Codex ko yahi ZIP
return karein; uske basis par next training notebook freeze hoga.

In [ ]:
import zipfile
report_files = [I12_ROOT/'iteration12_data_readiness.json', I12_ROOT/'iteration12_station_readiness.csv',
                I12_ROOT/'iteration12_decision.json', I12_ROOT/'metadata/imd_aws_mapping.receipt.json']
zip_path = I12_ROOT / 'SkyGuard_Iteration12_Genuine_IMD_AWS_Reports.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for path in report_files:
        if path.exists():
            z.write(path, path.relative_to(I12_ROOT))
print('Return this file:', zip_path)
from google.colab import files
files.download(str(zip_path))